# Northwind Analytics — End-to-End SQL Portfolio Project

**Database:** Northwind (SQLite)  
**Tools:** SQLite, Python (pandas, sqlalchemy)  
**Tracks:** Sales & Revenue · Customer RFM · Product Performance · Employee Performance

---

## Project Structure

```
northwind_analytics/
├── 0_setup/         — connect to the database
├── 1_staging/       — clean, renamed base views
├── 2_intermediate/  — joined & enriched models (inlined for SQLite)
├── 3_marts/         — business-ready aggregations
└── 4_analysis/      — insight queries & findings
```

## Schema Overview

| Table | Description |
|---|---|
| `Customers` | Who buys — 91 companies across 21 countries |
| `Orders` | When & where — 830 orders |
| `OrderDetails` | What & how much — 2,155 line items |
| `Products` | 77 products across 8 categories |
| `Categories` | Product groupings |
| `Suppliers` | 29 suppliers across 16 countries |
| `Employees` | 9 sales reps with reporting hierarchy |
| `Shippers` | 3 shipping companies |
| `Territories / Regions` | Sales geography |

---
## 0 — Setup

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine('sqlite:///Northwind.db')

def run(sql):
    """Execute SQL and return a DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

def execute(sql):
    """Execute SQL with no return (for CREATE / DROP statements)."""
    with engine.begin() as conn:
        for statement in sql.strip().split(';'):
            s = statement.strip()
            if s:
                conn.execute(text(s))

# Verify tables
run("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")

---
## 1 — Staging Layer

Clean, consistently named views on top of raw Northwind tables.  
No logic, no aggregations — just renaming, light casting, and documentation.

In [ ]:
execute("""
DROP VIEW IF EXISTS stg_customers;
CREATE VIEW stg_customers AS
SELECT
    CustomerID   AS customer_id,
    CompanyName  AS company_name,
    ContactName  AS contact_name,
    City         AS city,
    Country      AS country,
    Phone        AS phone
FROM Customers
""")

execute("""
DROP VIEW IF EXISTS stg_orders;
CREATE VIEW stg_orders AS
SELECT
    OrderID              AS order_id,
    CustomerID           AS customer_id,
    EmployeeID           AS employee_id,
    ShipVia              AS shipper_id,
    DATE(OrderDate)      AS order_date,
    DATE(RequiredDate)   AS required_date,
    DATE(ShippedDate)    AS shipped_date,
    ShipCountry          AS ship_country,
    ShipCity             AS ship_city,
    Freight              AS freight_amount
FROM Orders
""")

execute("""
DROP VIEW IF EXISTS stg_order_details;
CREATE VIEW stg_order_details AS
SELECT
    OrderID                                                     AS order_id,
    ProductID                                                   AS product_id,
    ROUND(UnitPrice, 2)                                         AS unit_price,
    Quantity                                                    AS quantity,
    Discount                                                    AS discount,
    ROUND(UnitPrice * Quantity * (1 - Discount), 2)             AS line_total
FROM OrderDetails
""")

execute("""
DROP VIEW IF EXISTS stg_products;
CREATE VIEW stg_products AS
SELECT
    ProductID            AS product_id,
    ProductName          AS product_name,
    CategoryID           AS category_id,
    SupplierID           AS supplier_id,
    ROUND(UnitPrice, 2)  AS unit_price,
    UnitsInStock         AS units_in_stock,
    Discontinued         AS is_discontinued
FROM Products
""")

execute("""
DROP VIEW IF EXISTS stg_employees;
CREATE VIEW stg_employees AS
SELECT
    EmployeeID                         AS employee_id,
    LastName || ', ' || FirstName      AS employee_name,
    Title                              AS title,
    ReportsTo                          AS manager_id,
    DATE(HireDate)                     AS hire_date
FROM Employees
""")

print('Staging views created.')

In [ ]:
# Sanity checks
checks = {
    'stg_customers':     ('SELECT COUNT(*) AS n FROM stg_customers',     91),
    'stg_orders':        ('SELECT COUNT(*) AS n FROM stg_orders',        830),
    'stg_order_details': ('SELECT COUNT(*) AS n FROM stg_order_details', 2155),
    'stg_products':      ('SELECT COUNT(*) AS n FROM stg_products',      77),
    'stg_employees':     ('SELECT COUNT(*) AS n FROM stg_employees',     9),
}

for view, (sql, expected) in checks.items():
    actual = run(sql)['n'][0]
    status = '✓' if actual == expected else '✗'
    print(f"{status} {view}: {actual} rows (expected {expected})")

---
## 2 — Mart Layer

Business-ready aggregations for each analysis track.  
Intermediate joins are inlined directly (SQLite does not reliably support view-on-view references).

### Mart 1 — Sales & Revenue

In [ ]:
execute("""
DROP VIEW IF EXISTS mart_sales_monthly;
CREATE VIEW mart_sales_monthly AS
SELECT
    STRFTIME('%Y',    o.OrderDate)    AS year,
    STRFTIME('%m',    o.OrderDate)    AS month,
    STRFTIME('%Y-%m', o.OrderDate)    AS year_month,
    COUNT(DISTINCT o.OrderID)         AS total_orders,
    ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 2)              AS total_revenue,
    ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)) /
          COUNT(DISTINCT o.OrderID), 2)                                         AS avg_order_value
FROM Orders o
JOIN OrderDetails od ON o.OrderID = od.OrderID
GROUP BY year, month
ORDER BY year, month
""")

run("SELECT * FROM mart_sales_monthly")

### Mart 2 — Customer RFM

In [ ]:
execute("""
DROP VIEW IF EXISTS mart_customer_rfm;
CREATE VIEW mart_customer_rfm AS
WITH last_order_date AS (
    SELECT MAX(DATE(OrderDate)) AS max_date FROM Orders
)
SELECT
    c.CustomerID                                                                AS customer_id,
    c.CompanyName                                                               AS customer_name,
    c.Country                                                                   AS customer_country,
    CAST(JULIANDAY((SELECT max_date FROM last_order_date))
       - JULIANDAY(MAX(DATE(o.OrderDate))) AS INT)                              AS recency_days,
    COUNT(DISTINCT o.OrderID)                                                   AS frequency,
    ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 2)              AS monetary_value,
    ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)) /
          COUNT(DISTINCT o.OrderID), 2)                                         AS avg_order_value
FROM Orders o
JOIN OrderDetails od ON o.OrderID    = od.OrderID
JOIN Customers     c ON o.CustomerID = c.CustomerID
GROUP BY c.CustomerID, c.CompanyName, c.Country
ORDER BY monetary_value DESC
""")

run("SELECT * FROM mart_customer_rfm LIMIT 10")

### Mart 3 — Product & Category Performance

In [ ]:
execute("""
DROP VIEW IF EXISTS mart_product_performance;
CREATE VIEW mart_product_performance AS
SELECT
    p.ProductID                                                                 AS product_id,
    p.ProductName                                                               AS product_name,
    cat.CategoryName                                                            AS category_name,
    sup.CompanyName                                                             AS supplier_name,
    sup.Country                                                                 AS supplier_country,
    COUNT(DISTINCT o.OrderID)                                                   AS times_ordered,
    SUM(od.Quantity)                                                            AS total_units_sold,
    ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 2)              AS total_revenue,
    ROUND(AVG(od.Discount) * 100, 1)                                            AS avg_discount_pct,
    p.Discontinued                                                              AS is_discontinued
FROM OrderDetails  od
JOIN Orders      o   ON od.OrderID   = o.OrderID
JOIN Products    p   ON od.ProductID = p.ProductID
JOIN Categories  cat ON p.CategoryID = cat.CategoryID
JOIN Suppliers   sup ON p.SupplierID = sup.SupplierID
GROUP BY p.ProductID, p.ProductName, cat.CategoryName,
         sup.CompanyName, sup.Country, p.Discontinued
ORDER BY total_revenue DESC
""")

run("SELECT * FROM mart_product_performance LIMIT 10")

### Mart 4 — Employee Performance

In [ ]:
execute("""
DROP VIEW IF EXISTS mart_employee_performance;
CREATE VIEW mart_employee_performance AS
SELECT
    e.EmployeeID                                                                AS employee_id,
    e.LastName || ', ' || e.FirstName                                           AS employee_name,
    e.Title                                                                     AS employee_title,
    COUNT(DISTINCT o.OrderID)                                                   AS total_orders,
    COUNT(DISTINCT o.CustomerID)                                                AS unique_customers,
    ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 2)              AS total_revenue,
    ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)) /
          COUNT(DISTINCT o.OrderID), 2)                                         AS avg_order_value,
    ROUND(AVG(
        CASE WHEN o.ShippedDate IS NOT NULL
             THEN JULIANDAY(o.ShippedDate) - JULIANDAY(o.OrderDate)
        END
    ), 1)                                                                       AS avg_days_to_ship
FROM Orders      o
JOIN OrderDetails  od ON o.OrderID    = od.OrderID
JOIN Employees     e  ON o.EmployeeID = e.EmployeeID
GROUP BY e.EmployeeID, e.LastName, e.FirstName, e.Title
ORDER BY total_revenue DESC
""")

run("SELECT * FROM mart_employee_performance")

---
## 3 — Analysis: Track 1 — Sales & Revenue

In [ ]:
# Q1: What is the overall revenue trend by month?
run("""
SELECT year_month, total_orders, total_revenue, avg_order_value
FROM mart_sales_monthly
ORDER BY year_month
""")

In [ ]:
# Q2: Which countries generate the most revenue?
run("""
SELECT
    c.Country                                                           AS country,
    COUNT(DISTINCT o.OrderID)                                           AS total_orders,
    ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 2)      AS total_revenue
FROM Orders o
JOIN OrderDetails od ON o.OrderID    = od.OrderID
JOIN Customers     c ON o.CustomerID = c.CustomerID
GROUP BY c.Country
ORDER BY total_revenue DESC
LIMIT 10
""")

In [ ]:
# Q3: How does revenue compare year-over-year?
run("""
SELECT
    year,
    COUNT(DISTINCT o.OrderID)                                           AS total_orders,
    ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 2)      AS total_revenue
FROM Orders o
JOIN OrderDetails od ON o.OrderID = od.OrderID
GROUP BY year
ORDER BY year
""")

---
## 4 — Analysis: Track 2 — Customer Behavior

In [ ]:
# Q4: Who are the top 10 customers by lifetime value?
run("""
SELECT customer_name, customer_country, frequency, monetary_value, recency_days
FROM mart_customer_rfm
LIMIT 10
""")

In [ ]:
# Q5: RFM segmentation — classify customers into tiers
run("""
SELECT
    customer_name,
    recency_days,
    frequency,
    monetary_value,
    CASE
        WHEN recency_days <= 30  AND frequency >= 5 AND monetary_value >= 5000 THEN 'Champions'
        WHEN recency_days <= 90  AND frequency >= 3                            THEN 'Loyal'
        WHEN recency_days <= 90  AND frequency < 3                             THEN 'Promising'
        WHEN recency_days BETWEEN 91 AND 180                                   THEN 'At Risk'
        ELSE 'Lost'
    END AS rfm_segment
FROM mart_customer_rfm
ORDER BY monetary_value DESC
""")

In [ ]:
# Q6: How many customers fall into each RFM segment?
run("""
SELECT
    CASE
        WHEN recency_days <= 30  AND frequency >= 5 AND monetary_value >= 5000 THEN 'Champions'
        WHEN recency_days <= 90  AND frequency >= 3                            THEN 'Loyal'
        WHEN recency_days <= 90  AND frequency < 3                             THEN 'Promising'
        WHEN recency_days BETWEEN 91 AND 180                                   THEN 'At Risk'
        ELSE 'Lost'
    END                          AS rfm_segment,
    COUNT(*)                     AS customer_count,
    ROUND(SUM(monetary_value),2) AS segment_revenue
FROM mart_customer_rfm
GROUP BY rfm_segment
ORDER BY segment_revenue DESC
""")

---
## 5 — Analysis: Track 3 — Product & Category Performance

In [ ]:
# Q7: What are the top 10 products by revenue?
run("""
SELECT product_name, category_name, total_units_sold, total_revenue, avg_discount_pct
FROM mart_product_performance
LIMIT 10
""")

In [ ]:
# Q8: Which category drives the most revenue?
run("""
SELECT
    category_name,
    COUNT(*)                         AS product_count,
    SUM(total_units_sold)            AS total_units_sold,
    ROUND(SUM(total_revenue), 2)     AS total_revenue,
    ROUND(AVG(avg_discount_pct), 1)  AS avg_discount_pct
FROM mart_product_performance
GROUP BY category_name
ORDER BY total_revenue DESC
""")

In [ ]:
# Q9: Are discontinued products still contributing revenue?
run("""
SELECT
    CASE WHEN is_discontinued = 1 THEN 'Discontinued' ELSE 'Active' END AS status,
    COUNT(*)                      AS product_count,
    ROUND(SUM(total_revenue), 2)  AS total_revenue,
    ROUND(AVG(avg_discount_pct),1)AS avg_discount_pct
FROM mart_product_performance
GROUP BY is_discontinued
""")

---
## 6 — Analysis: Track 4 — Employee Performance

In [ ]:
# Q10: Who is the top-performing sales rep?
run("""
SELECT employee_name, employee_title, total_orders,
       unique_customers, total_revenue, avg_order_value, avg_days_to_ship
FROM mart_employee_performance
""")

In [ ]:
# Q11: Revenue per employee by year
run("""
SELECT
    e.LastName || ', ' || e.FirstName                                    AS employee_name,
    STRFTIME('%Y', o.OrderDate)                                          AS year,
    ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 2)       AS annual_revenue
FROM Orders o
JOIN OrderDetails od ON o.OrderID    = od.OrderID
JOIN Employees     e  ON o.EmployeeID = e.EmployeeID
GROUP BY e.EmployeeID, year
ORDER BY year, annual_revenue DESC
""")

In [ ]:
# Q12: Which shipper is fastest on average?
run("""
SELECT
    s.CompanyName                                           AS shipper,
    COUNT(o.OrderID)                                        AS total_shipments,
    ROUND(AVG(
        CASE WHEN o.ShippedDate IS NOT NULL
             THEN JULIANDAY(o.ShippedDate) - JULIANDAY(o.OrderDate)
        END
    ), 1)                                                   AS avg_days_to_ship
FROM Orders   o
JOIN Shippers s ON o.ShipVia = s.ShipperID
GROUP BY s.ShipperID
ORDER BY avg_days_to_ship
""")

---
## 7 — Key Findings (README Summary)

Fill in after running all queries above.

| # | Finding | Track |
|---|---|---|
| 1 | Revenue peaked in **[month/year]**, driven by **[category]** | Sales |
| 2 | Top 10 customers account for **~X%** of total revenue | Customers |
| 3 | **[X]%** of customers are classified as 'At Risk' or 'Lost' | Customers |
| 4 | **[Product]** is the highest-revenue product; **[Category]** leads by category | Products |
| 5 | Discontinued products still contributed **$X** in revenue | Products |
| 6 | **[Employee]** leads in total revenue; **[Employee]** has the fastest ship time | Employees |
| 7 | **[Shipper]** averages the fastest delivery at **X days** | Operations |